# Multi-hop reasoning

Most questions that feel "hard" are not hard because any single fact is obscure - they are hard because the answer depends on chaining several facts together, where we cannot know what to look for next until we have found the previous piece. "Which country does the highest peak of the mountain range forming the border between Europe and Asia belong to?" requires knowing which mountain range forms that border before we can ask about its highest peak, and knowing the peak before we can ask which country it is in. Each answer gates the next question.

Multi-hop reasoning formalises this chain. Given a complex question, the system first decomposes it into the initial sub-questions whose answers are prerequisites for everything else. It retrieves those facts, then asks: given what we know now, what else do we still need? It continues retrieving until the accumulated facts are sufficient to answer the original question, then synthesises a final answer from everything gathered.

The transparency is the core benefit. Rather than asking the LLM to jump directly to an answer - which works well for simple questions but degrades on deeply nested ones - multi-hop reasoning produces a traceable chain where every intermediate result is visible and verifiable. This notebook implements each step of that chain one function at a time.

In [1]:
import os
from collections import namedtuple
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0 keeps decomposition and follow-up decisions deterministic;
# a small amount of variation in fact retrieval is acceptable (0.3)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.3)

## Representing a retrieved fact
Every piece of information collected during the reasoning chain comes from answering a specific sub-question at a specific hop. We need to carry those three things together - the hop number, the sub-question that was asked, and the answer that was retrieved - throughout the pipeline. A `namedtuple` with three fields is the right level of structure: it is immutable (facts do not change once retrieved), it supports attribute access by name, and it adds zero boilerplate. The hop number is stored on the record itself rather than being implicit in a list index, because the synthesis step needs to present the chain in order and attribute access is more readable than `record[0]`.

In [3]:
# Three fields — all populated at retrieval time, never modified afterwards
Hop = namedtuple('Hop', ['number', 'query', 'fact'])
# number — 1-based hop index (hop 1 = first retrieval, hop 2 = follow-up, ...)
# query  — the sub-question that was asked at this hop
#           e.g. "What mountain range forms the border between Europe and Asia?"
# fact   — the retrieved answer to that sub-question
#           e.g. "The Ural Mountains form the primary border between Europe and Asia."

The `Hop` record is the only data structure in this implementation. The full reasoning chain is a `List[Hop]` accumulated across iterations, and the driver function returns that list alongside the final answer in a plain dict.

## Decomposing the question into initial sub-queries
The first step is not retrieval - it is planning. Before we can retrieve any facts we need to know which facts to retrieve first. For a simple question this might be just one sub-query; for a deeply nested one it might be two or three. The key constraint is that these initial sub-queries must be answerable without knowing anything else - they are the roots of the reasoning tree, not the leaves. Asking "what is the highest peak of the mountain range that forms the Europe-Asia border?" as a first query fails because answering it requires knowing the mountain range, which is itself something we need to retrieve.

In [4]:
def decompose_question(question: str) -> List[str]:
    """Return 1-3 initial sub-queries whose answers are prerequisites for the full answer.

    These queries must be independently answerable — they cannot rely on information
    that has not yet been retrieved.
    """
    prompt = (
        f"Given the following complex question, identify 1-3 initial sub-questions "
        f"that must be answered first as stepping stones toward the full answer.\n\n"
        f"Question: {question}\n\n"
        f"Each sub-question must be:\n"
        f"  - Independently answerable (no prior context needed)\n"
        f"  - Specific enough to yield a concrete fact\n"
        f"  - A necessary prerequisite for answering the full question\n\n"
        f"Output only a numbered list, nothing else.\n\n"
        f"Format:\n"
        f"1. First sub-question\n"
        f"2. Second sub-question"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    queries = []
    for line in raw.splitlines():
        line = line.strip()
        # Match "1. ...", "2. ..." — digit at start, period present
        if line and line[0].isdigit() and '.' in line:
            query = line.split('.', 1)[1].strip()   # text after "N."
            if query:
                queries.append(query)
    return queries

The prompt's three-bullet constraint - independently answerable, specific, necessary - does meaningful work. Without "independently answerable", the model may generate sub-questions that presuppose answers to other sub-questions, creating a circular dependency at the very first step. The parser matches any line starting with a digit and containing a period, then splits on the first period only so that sub-questions containing abbreviations (e.g. "U.S. GDP") are not truncated.

In [5]:
# Test with a question that clearly requires several intermediate facts
test_question = (
    "Which country does the highest peak of the mountain range "
    "that forms the border between Europe and Asia belong to?"
)
test_queries = decompose_question(test_question)

print(f"Question: {test_question}\n")
print(f"Initial sub-queries ({len(test_queries)}):")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

Question: Which country does the highest peak of the mountain range that forms the border between Europe and Asia belong to?

Initial sub-queries (3):
  1. What is the name of the mountain range that forms the border between Europe and Asia?
  2. What is the name of the highest peak in that mountain range?
  3. Which country is home to the highest peak of that mountain range?


## Retrieving a fact for a single sub-query
Each sub-query is answered individually, with all previously retrieved facts passed as context. The context serves two purposes: it prevents the model from giving an answer that contradicts something already established, and it allows the model to use earlier findings when they are relevant (for example, answering "what is the highest peak of the Ural Mountains?" is easier if the context already confirms we are talking about the Ural Mountains specifically). On the first hop, the context is empty - no prior facts exist yet. On subsequent hops, every `Hop` retrieved so far is formatted into the context block.

In [6]:
def retrieve_fact(question: str, query: str, prior_hops: List[Hop]) -> str:
    """Answer a single sub-query using the LLM's knowledge and all prior hops as context.

    In a production system this is where external retrieval (a search engine, a vector
    store, a database) would be called instead of relying on the LLM's parametric memory.
    The interface is the same either way: query in, fact string out.
    """
    # Format prior facts as a compact context block; empty string on the first hop
    if prior_hops:
        context = "\n".join(
            f"  Hop {h.number}: {h.fact}"   # one line per previously retrieved fact
            for h in prior_hops
        )
    else:
        context = "  (none — this is the first retrieval)"

    prompt = (
        f"You are answering one sub-question as part of a multi-step reasoning chain.\n\n"
        f"Original question: {question}\n\n"
        f"Facts established so far:\n{context}\n\n"
        f"Sub-question to answer now: {query}\n\n"
        f"Give a concise, factual answer to this sub-question only. "
        f"One to two sentences maximum."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

The docstring's production note is intentional: the function's signature - `(question, query, prior_hops) → str` - is implementation-agnostic. Swapping the LLM call for a retrieval API call requires changing only the two lines inside the `if prior_hops` block and the `llm.invoke` call; the rest of the pipeline is unaffected. The context block uses attribute access (`h.number`, `h.fact`) on the `Hop` namedtuple rather than index access (`h[0]`, `h[2]`), which keeps the intent readable as the context grows across hops.

In [7]:
# Test on the first query from the decompose step — no prior context on hop 1
if test_queries:
    test_fact = retrieve_fact(test_question, test_queries[0], prior_hops=[])
    print(f"Sub-query: {test_queries[0]}\n")
    print(f"Retrieved fact:\n{test_fact}")

Sub-query: What is the name of the mountain range that forms the border between Europe and Asia?

Retrieved fact:
The mountain range that forms the border between Europe and Asia is called the Ural Mountains.


## Deciding whether to continue - identifying follow-up queries
After each retrieval round the system must decide whether to stop or continue. If the accumulated facts are enough to answer the original question, doing more hops wastes calls and can introduce noise. If they are not enough, the system needs to generate follow-up queries that are specifically motivated by what was just learned - this is what makes each hop 'hop': the next query is only knowable because of the previous answer.

The stopping condition is explicit: the model responds with the word `SUFFICIENT` when no more information is needed. This is cleaner than asking for a boolean because the same prompt can return either a list of follow-up queries or the stopping signal, with no second call required. The empty-list return value signals to the driver that it should exit the loop.

In [8]:
def identify_followup_queries(question: str, hops_so_far: list) -> list:
    # Build a compact summary of every fact retrieved so far
    facts_summary = "\n".join(
        f"  Hop {h.number} — Q: {h.query}\n             A: {h.fact}"
        for h in hops_so_far
    )

    prompt = (
        f"Assess whether the facts gathered so far are sufficient to fully answer "
        f"the original question, or whether additional information is still needed.\n\n"
        f"Original question: {question}\n\n"
        f"Facts gathered:\n{facts_summary}\n\n"
        f"If sufficient: respond with exactly the word SUFFICIENT on its own line.\n"
        f"If not sufficient: list 1-2 specific follow-up sub-questions as a numbered list. "
        f"Each follow-up must be motivated by and build on the facts above."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    # Check for the stopping signal first — 'in upper()' catches 'SUFFICIENT.' or 'SUFFICIENT!'
    if 'SUFFICIENT' in raw.upper():
        return []   # empty list = driver should stop iterating

    # Parse numbered follow-up queries using the same pattern as decompose_question
    queries = []
    for line in raw.splitlines():
        line = line.strip()
        if line and line[0].isdigit() and '.' in line:
            query = line.split('.', 1)[1].strip()   # text after 'N.'
            if query:
                queries.append(query)
    return queries

The `SUFFICIENT` check uses `in raw.upper()` rather than exact equality so that responses like `SUFFICIENT — no more information is needed.` still trigger the stopping condition. The follow-up query parser is identical to the one in `decompose_question`: both use `line[0].isdigit() and '.' in line`. If the model returns `1) ...` instead of `1. ...`, neither parser matches and the function returns an empty list - which safely triggers the stopping condition rather than crashing or entering an infinite loop.

In [9]:
# Simulate a one-hop context using the fact retrieved in the previous cell
if test_queries and test_fact:
    one_hop = [Hop(number=1, query=test_queries[0], fact=test_fact)]
    follow_ups = identify_followup_queries(test_question, one_hop)

    if follow_ups:
        print(f"Follow-up queries needed ({len(follow_ups)}):")
        for i, q in enumerate(follow_ups, 1):
            print(f"  {i}. {q}")
    else:
        print("Sufficient after one hop — no follow-ups needed.")

Follow-up queries needed (2):
  1. What is the name of the highest peak in the Ural Mountains?
  2. Which country does the highest peak of the Ural Mountains belong to?


## Synthesising the final answer
Once the driver determines that enough facts have been collected - either because the follow-up function returned an empty list or because `max_hops` was reached - the synthesis step takes the complete `List[Hop]` and composes a final answer to the original question. The synthesis prompt presents the full reasoning chain in order, labelling each hop's query and fact, and asks the model to write a direct, coherent answer that integrates all of them. This is different from just returning the last fact: it forces the model to connect the chain explicitly, which often surfaces conclusions that are not visible in any single hop.

In [10]:
def synthesize_answer(question: str, hops: list) -> str:
    # Format the chain: each hop as 'Hop N — Q: ... / A: ...'
    chain = "\n".join(
        f"  Hop {h.number} — Q: {h.query}\n"
        f"              A: {h.fact}"
        for h in hops
    )

    prompt = (
        f"Using the following step-by-step reasoning chain, provide a comprehensive "
        f"answer to the original question.\n\n"
        f"Original question: {question}\n\n"
        f"Reasoning chain:\n{chain}\n\n"
        f"Synthesise the chain into a clear, direct answer. "
        f"Do not repeat the individual hops — just state the conclusion."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

The instruction `Do not repeat the individual hops — just state the conclusion` prevents the model from producing an answer that reads like a transcript of the chain (e.g. "First we found X, then we found Y, therefore Z"). The goal is a clean conclusion, not a narration. The chain itself is already visible in `result["hops"]` for inspection; the synthesis step produces the final answer, not the trace.

## The complete multi-hop pipeline
All four building blocks are now assembled into a single driver that manages the iteration. Hop 1 is the only one that uses `decompose_question`; all subsequent hops use `identify_followup_queries` instead. The `max_hops` guard prevents open-ended chains. The driver collects every `Hop` in a single flat list that records the full reasoning trace, regardless of which iteration produced each hop. Python's `for-else` construct cleanly distinguishes the two exit conditions: `break` when sufficient facts are gathered, `else` when the loop exhausts `max_hops`.

In [11]:
def multi_hop_reason(question: str, max_hops: int = 4) -> dict:
    print(f"Question: {question}\n")
    all_hops = []   # flat list of Hop namedtuples — the full reasoning chain

    # ── Hop 1: decompose and retrieve initial facts ────────────────────
    print("Step 1 — Decomposing into initial sub-queries...")
    initial_queries = decompose_question(question)
    for i, q in enumerate(initial_queries, 1):
        print(f"  {i}. {q}")

    print("\nHop 1 — Retrieving initial facts...")
    for query in initial_queries:
        fact = retrieve_fact(question, query, prior_hops=[])   # no prior context
        all_hops.append(Hop(number=1, query=query, fact=fact))
        print(f"  -> {fact[:90]}...")

    # ── Hops 2+: follow-up until SUFFICIENT or max_hops reached ────────
    for hop_num in range(2, max_hops + 1):
        print(f"\nChecking whether more information is needed (hop {hop_num})...")
        followup_queries = identify_followup_queries(question, all_hops)

        if not followup_queries:
            # Empty list = stopping signal; accumulated facts are sufficient
            print(f"  Sufficient. Stopping after hop {hop_num - 1}.")
            break

        print(f"  {len(followup_queries)} follow-up query/queries identified.")
        print(f"\nHop {hop_num} — Retrieving follow-up facts...")
        for query in followup_queries:
            # All hops so far are passed as context for this retrieval
            fact = retrieve_fact(question, query, prior_hops=all_hops)
            all_hops.append(Hop(number=hop_num, query=query, fact=fact))
            print(f"  -> {fact[:90]}...")
    else:
        # for-else: runs only when the loop exits by exhausting max_hops (no break)
        print(f"\nReached max_hops ({max_hops}).")

    # ── Synthesis: compose the final answer from all gathered hops ─────
    print("\nSynthesising final answer...")
    final_answer = synthesize_answer(question, all_hops)

    return {
        "question":     question,
        "hops":         all_hops,     # List[Hop] — complete reasoning chain
        "final_answer": final_answer,
    }

The `for-else` construct is a deliberate Python idiom: the `else` block executes only when the loop finishes by exhausting its range, not when it exits via `break`. This cleanly distinguishes `stopped because sufficient` from `stopped because hit the limit` without a separate flag variable. The flat `all_hops` list is the complete record of the reasoning chain in retrieval order - the same order that the synthesis prompt presents it in.

## Applying the pipeline to a multi-hop question
The question is chosen because it structurally requires chaining: we cannot ask about the highest peak before knowing which mountain range forms the border, and we cannot ask which country the peak belongs to before knowing the peak. A direct answer may guess correctly from training data, but the multi-hop version makes every intermediate step explicit and individually checkable.

In [12]:
question = (
    "Which country does the highest peak of the mountain range "
    "that forms the border between Europe and Asia belong to?"
)

result = multi_hop_reason(question)

print("\n" + "=" * 65)
print("FULL REASONING CHAIN")
print("=" * 65)
for h in result["hops"]:
    print(f"\n  Hop {h.number}:")
    print(f"    Q: {h.query}")
    print(f"    A: {h.fact}")

print("\n" + "=" * 65)
print("FINAL ANSWER")
print("=" * 65)
print(result["final_answer"])

Question: Which country does the highest peak of the mountain range that forms the border between Europe and Asia belong to?

Step 1 — Decomposing into initial sub-queries...
  1. What is the name of the mountain range that forms the border between Europe and Asia?
  2. What is the name of the highest peak in that mountain range?
  3. Which country is home to that highest peak?

Hop 1 — Retrieving initial facts...
  -> The mountain range that forms the border between Europe and Asia is called the Ural Mounta...
  -> The highest peak in the mountain range that forms the border between Europe and Asia is Mo...
  -> The highest peak of the mountain range that forms the border between Europe and Asia, Moun...

Checking whether more information is needed (hop 2)...
  2 follow-up query/queries identified.

Hop 2 — Retrieving follow-up facts...
  -> Mount Elbrus is not part of the Ural Mountains; it is part of the Caucasus Mountain range....
  -> Mount Elbrus has a height of 5,642 meters (18,

The `result["hops"]` list exposes each step of the chain independently. Each `Hop` record carries `.number`, `.query`, and `.fact` accessible by name - useful for logging, building a citation trail, or debugging a chain that produced an incorrect final answer.

## Comparing with direct answering
The clearest illustration of multi-hop's value is a question that involves two separate attribution chains that must be compared. A direct call may produce the right answer from memorised training data, but it will not show how it got there. The multi-hop version makes both chains visible, which makes the comparison auditable — you can check each retrieved fact independently rather than trusting the final answer blindly.

In [13]:
def direct_answer(question: str) -> str:
    # Single call — no decomposition, no intermediate steps, no trace
    response = llm.invoke([HumanMessage(
        content=f"Answer this question as accurately as possible: {question}"
    )])
    return response.content.strip()


comparison_q = (
    "Was the first commercially available touchscreen phone made by "
    "the same company that created the first mass-market GUI computer?"
)

print("--- Direct answer (1 LLM call) ---")
direct = direct_answer(comparison_q)
print(direct)

print("\n" + "-" * 65)
print("--- Multi-hop reasoning ---")
result = multi_hop_reason(comparison_q)

print("\nReasoning chain:")
for h in result["hops"]:
    print(f"  Hop {h.number}: {h.query}")
    print(f"    -> {h.fact[:80]}...")

print(f"\nFinal answer:\n{result['final_answer']}")

--- Direct answer (1 LLM call) ---
Yes, the first commercially available touchscreen phone was made by IBM, which is the same company that created the first mass-market GUI computer, the IBM PC with the Windows operating system. The first commercially available touchscreen phone was the IBM Simon Personal Communicator, released in 1994. The IBM PC, along with the Windows operating system, played a significant role in popularizing graphical user interfaces in the personal computing market.

-----------------------------------------------------------------
--- Multi-hop reasoning ---
Question: Was the first commercially available touchscreen phone made by the same company that created the first mass-market GUI computer?

Step 1 — Decomposing into initial sub-queries...
  1. What company created the first mass-market GUI computer?
  2. What was the first commercially available touchscreen phone?
  3. Which company manufactured the first commercially available touchscreen phone?

Hop 1 — R

**When to use multi-hop reasoning:** Questions where each sub-answer genuinely gates the next sub-question - we cannot know what to ask second until we have answered the first. If a question can be answered with a single retrieval step, multi-hop adds latency without adding quality.

**The stopping condition is the most important design decision.** The `SUFFICIENT` signal controls how many calls the pipeline makes. An overly conservative model (always says more is needed) drives up cost; an overly aggressive one (stops too early) produces incomplete answers.

**Production extension: replace `retrieve_fact` with real retrieval.** The function's signature `(question, query, prior_hops) → str` works equally well whether the body calls an LLM, a search engine, or a vector store. Structuring the pipeline around this interface means adding real retrieval is a one-function change.

**Limitations:** Multiple sequential LLM calls make multi-hop slower than direct answering. Quality depends on decomposition at step 1 and follow-up quality at each subsequent step - if either generates a poor sub-question, the error propagates forward through the chain.